# RD4AD - WideResNet50-2, x224

Reverse Distillation for Anomaly Detection using a frozen WideResNet50-2 encoder
and a trainable three-scale decoder. Trained exclusively on normal wafers.

**Default mode** (`RETRAIN = False`): loads the saved checkpoint and reuses
saved evaluation / UMAP artifacts when available.

**Retrain mode** (`RETRAIN = True`): runs `scripts/train_rd4ad.py` with the
checked-in `train_config.toml`, then re-evaluates.

## Imports and Paths

In [ ]:
try:
    from pathlib import Path
    import json
    import subprocess
    import sys
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    from IPython.display import Image as IPyImage, display
    from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
    from torch.utils.data import DataLoader
    cwd = Path.cwd().resolve()
    REPO_ROOT = None
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'src' / 'wafer_defect').exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None:
        raise RuntimeError('Could not find repo root containing src/wafer_defect')
    SRC_ROOT = REPO_ROOT / 'src'
    if str(SRC_ROOT) not in sys.path:
        sys.path.insert(0, str(SRC_ROOT))
    EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'anomaly_detection' / 'rd4ad' / 'wideresnet50' / 'x224' / 'main'
    TRAIN_CONFIG = EXPERIMENT_DIR / 'train_config.toml'
    ARTIFACT_DIR = EXPERIMENT_DIR / 'artifacts' / 'rd4ad_wrn50_x224'
    RESULTS_DIR = ARTIFACT_DIR / 'results'
    PLOTS_DIR = ARTIFACT_DIR / 'plots'
    UMAP_DIR = RESULTS_DIR / 'umap'
    CHECKPOINT_PATH = ARTIFACT_DIR / 'checkpoints' / 'best_model.pt'
    METADATA_PATH = REPO_ROOT / 'data' / 'processed' / 'x224' / 'wm811k' / 'metadata_50k_5pct.csv'
    RUNNER_SCRIPT = REPO_ROOT / 'scripts' / 'train_rd4ad.py'
    RETRAIN = False
    REGENERATE_UMAP = False
    NUM_WORKERS = 8
    DEVICE_NAME = 'auto'
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    UMAP_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Repo root : {REPO_ROOT}')
    print(f'Artifact  : {ARTIFACT_DIR}')
    print(f'Checkpoint: {CHECKPOINT_PATH} (exists={CHECKPOINT_PATH.exists()})')
    print(f'UMAP dir   : {UMAP_DIR}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from pathlib import path\nimport json\nimport subprocess\nimport sys\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom ipython.display import image as ipyimage, display\nfrom sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score\nfrom torch.utils.data import dataloader\ncwd = path.cwd().resolve()\nrepo_root = none\nfor candidate in [cwd, *cwd.parents]:\n    if (candidate / 'src' / 'wafer_defect').exists():\n        repo_root = candidate\n        break\nif repo_root is none:\n    raise runtimeerror('could not find repo root containing src/wafer_defect')\nsrc_root = repo_root / 'src'\nif str(src_root) not in sys.path:\n    sys.path.insert(0, str(src_root))\nexperiment_dir = repo_root / 'experiments' / 'anomaly_detection' / 'rd4ad' / 'wideresnet50' / 'x224' / 'main'\ntrain_config = experiment_dir / 'train_config.toml'\nartifact_dir = experiment_dir / 'artifacts' / 'rd4ad_wrn50_x224'\nresults_dir = artifact_dir / 'results'\nplots_dir = artifact_dir / 'plots'\numap_dir = results_dir / 'umap'\ncheckpoint_path = artifact_dir / 'checkpoints' / 'best_model.pt'\nmetadata_path = repo_root / 'data' / 'processed' / 'x224' / 'wm811k' / 'metadata_50k_5pct.csv'\nrunner_script = repo_root / 'scripts' / 'train_rd4ad.py'\nretrain = false\nregenerate_umap = false\nnum_workers = 8\ndevice_name = 'auto'\nartifact_dir.mkdir(parents=true, exist_ok=true)\nresults_dir.mkdir(parents=true, exist_ok=true)\nplots_dir.mkdir(parents=true, exist_ok=true)\numap_dir.mkdir(parents=true, exist_ok=true)\nprint(f'repo root : {repo_root}')\nprint(f'artifact  : {artifact_dir}')\nprint(f'checkpoint: {checkpoint_path} (exists={checkpoint_path.exists()})')\nprint(f'umap dir   : {umap_dir}')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Optional Retraining

With `RETRAIN = False` the notebook skips directly to loading the saved checkpoint.

In [ ]:
try:
    if RETRAIN:
        cmd = [sys.executable, '-u', str(RUNNER_SCRIPT), '--config', str(TRAIN_CONFIG)]
        print('Launching training:', ' '.join(cmd))
        subprocess.run(cmd, check=True, cwd=REPO_ROOT)
    else:
        print('RETRAIN=False - skipping training, loading checkpoint.')
        if not CHECKPOINT_PATH.exists():
            print(f'WARNING: checkpoint not found at {CHECKPOINT_PATH}.\nSet RETRAIN=True to train from scratch.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "if retrain:\n    cmd = [sys.executable, '-u', str(runner_script), '--config', str(train_config)]\n    print('launching training:', ' '.join(cmd))\n    subprocess.run(cmd, check=true, cwd=repo_root)\nelse:\n    print('retrain=false - skipping training, loading checkpoint.')\n    if not checkpoint_path.exists():\n        print(f'warning: checkpoint not found at {checkpoint_path}.\\nset retrain=true to train from scratch.')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Load Config and Metadata

In [ ]:
try:
    from wafer_defect.config import load_toml
    from wafer_defect.data.wm811k import WaferMapDataset
    config = load_toml(str(TRAIN_CONFIG))
    image_size = int(config['data'].get('image_size', 224))
    batch_size = int(config['data'].get('batch_size', 64))
    threshold_quantile = float(config['scoring'].get('threshold_quantile', 0.95))

    def resolve_device(name: str) -> torch.device:
        if name == 'auto':
            return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        return torch.device(name)
    device = resolve_device(DEVICE_NAME)
    print(f'Device: {device}  |  image_size: {image_size}  |  threshold_quantile: {threshold_quantile}')
    metadata = pd.read_csv(METADATA_PATH)
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from wafer_defect.config import load_toml\nfrom wafer_defect.data.wm811k import wafermapdataset\nconfig = load_toml(str(train_config))\nimage_size = int(config['data'].get('image_size', 224))\nbatch_size = int(config['data'].get('batch_size', 64))\nthreshold_quantile = float(config['scoring'].get('threshold_quantile', 0.95))\n\ndef resolve_device(name: str) -> torch.device:\n    if name == 'auto':\n        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n    return torch.device(name)\ndevice = resolve_device(device_name)\nprint(f'device: {device}  |  image_size: {image_size}  |  threshold_quantile: {threshold_quantile}')\nmetadata = pd.read_csv(metadata_path)\ndisplay(metadata['split'].value_counts().rename_axis('split').to_frame('count'))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Load Model from Checkpoint

In [ ]:
try:
    from wafer_defect.models.rd4ad import build_rd4ad_from_config
    model = build_rd4ad_from_config(config).to(device)
    if CHECKPOINT_PATH.exists():
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        best_epoch = checkpoint.get('best_epoch', '?')
        best_val_loss = checkpoint.get('best_val_loss', float('nan'))
        print(f'Loaded checkpoint | best_epoch={best_epoch} | best_val_loss={best_val_loss:.6f}')
    else:
        print('No checkpoint found - model weights are random.')
    model.eval()
    trainable = sum((p.numel() for p in model.parameters() if p.requires_grad))
    total = sum((p.numel() for p in model.parameters()))
    print(f'Trainable parameters: {trainable:,} / {total:,}')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from wafer_defect.models.rd4ad import build_rd4ad_from_config\nmodel = build_rd4ad_from_config(config).to(device)\nif checkpoint_path.exists():\n    checkpoint = torch.load(checkpoint_path, map_location=device)\n    model.load_state_dict(checkpoint['model_state_dict'])\n    best_epoch = checkpoint.get('best_epoch', '?')\n    best_val_loss = checkpoint.get('best_val_loss', float('nan'))\n    print(f'loaded checkpoint | best_epoch={best_epoch} | best_val_loss={best_val_loss:.6f}')\nelse:\n    print('no checkpoint found - model weights are random.')\nmodel.eval()\ntrainable = sum((p.numel() for p in model.parameters() if p.requires_grad))\ntotal = sum((p.numel() for p in model.parameters()))\nprint(f'trainable parameters: {trainable:,} / {total:,}')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Training History

In [ ]:
try:
    history_path = RESULTS_DIR / 'history.json'
    if history_path.exists():
        history = json.loads(history_path.read_text(encoding='utf-8'))
        history_df = pd.DataFrame(history)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')
        axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Total cosine loss')
        axes[0].legend()
        for col, label in [('train_l1', 'l1 (layer1)'), ('train_l2', 'l2 (layer2)'), ('train_l3', 'l3 (layer3)')]:
            axes[1].plot(history_df['epoch'], history_df[col], label=label)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Per-scale train losses')
        axes[1].legend()
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        display(history_df.tail(5))
    else:
        print('No history.json found - training has not been run yet.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "history_path = results_dir / 'history.json'\nif history_path.exists():\n    history = json.loads(history_path.read_text(encoding='utf-8'))\n    history_df = pd.dataframe(history)\n    fig, axes = plt.subplots(1, 2, figsize=(14, 4))\n    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train')\n    axes[0].plot(history_df['epoch'], history_df['val_loss'], label='val')\n    axes[0].set_xlabel('epoch')\n    axes[0].set_ylabel('loss')\n    axes[0].set_title('total cosine loss')\n    axes[0].legend()\n    for col, label in [('train_l1', 'l1 (layer1)'), ('train_l2', 'l2 (layer2)'), ('train_l3', 'l3 (layer3)')]:\n        axes[1].plot(history_df['epoch'], history_df[col], label=label)\n    axes[1].set_xlabel('epoch')\n    axes[1].set_ylabel('loss')\n    axes[1].set_title('per-scale train losses')\n    axes[1].legend()\n    plt.tight_layout()\n    fig.savefig(plots_dir / 'training_history.png', dpi=150, bbox_inches='tight')\n    plt.show()\n    display(history_df.tail(5))\nelse:\n    print('no history.json found - training has not been run yet.')"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Score Inference

Runs the model forward pass on the val and test splits to produce per-wafer anomaly scores.

In [ ]:
@torch.no_grad()
def infer_scores(split: str) -> tuple[np.ndarray, np.ndarray]:
    """Return (labels, scores) arrays for the given split."""
    dataset = WaferMapDataset(str(METADATA_PATH), split=split, image_size=image_size)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    all_scores, all_labels = ([], [])
    for images, labels in loader:
        images = images.to(device)
        scores = model(images).cpu().numpy()
        all_scores.append(scores)
        all_labels.append(labels.numpy())
    return (np.concatenate(all_labels), np.concatenate(all_scores))
val_labels, val_scores = infer_scores('val')
test_labels, test_scores = infer_scores('test')
print(f'Val  : {len(val_labels):,} samples  |  anomaly rate={val_labels.mean():.3%}')
print(f'Test : {len(test_labels):,} samples  |  anomaly rate={test_labels.mean():.3%}')

## Threshold and Metrics

In [ ]:
try:
    from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics
    val_normal_scores = val_scores[val_labels == 0]
    threshold = float(np.quantile(val_normal_scores, threshold_quantile))
    print(f'Val-normal {threshold_quantile:.0%} threshold: {threshold:.6f}')
    val_metrics = summarize_threshold_metrics(val_labels, val_scores, threshold)
    test_metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)
    metrics_df = pd.DataFrame([{'split': 'val', **{k: v for k, v in val_metrics.items() if k != 'confusion_matrix'}}, {'split': 'test', **{k: v for k, v in test_metrics.items() if k != 'confusion_matrix'}}])
    display(metrics_df.set_index('split'))
    pd.DataFrame({'label': val_labels, 'score': val_scores}).to_csv(RESULTS_DIR / 'val_scores.csv', index=False)
    pd.DataFrame({'label': test_labels, 'score': test_scores}).to_csv(RESULTS_DIR / 'test_scores.csv', index=False)
    summary = {'threshold': threshold, 'threshold_quantile': threshold_quantile, 'val': val_metrics, 'test': test_metrics}
    (RESULTS_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"\nTest F1={test_metrics['f1']:.4f}  AUROC={test_metrics['auroc']:.4f}  AUPRC={test_metrics['auprc']:.4f}")
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'from wafer_defect.evaluation.reconstruction_metrics import summarize_threshold_metrics, sweep_threshold_metrics\nval_normal_scores = val_scores[val_labels == 0]\nthreshold = float(np.quantile(val_normal_scores, threshold_quantile))\nprint(f\'val-normal {threshold_quantile:.0%} threshold: {threshold:.6f}\')\nval_metrics = summarize_threshold_metrics(val_labels, val_scores, threshold)\ntest_metrics = summarize_threshold_metrics(test_labels, test_scores, threshold)\nmetrics_df = pd.dataframe([{\'split\': \'val\', **{k: v for k, v in val_metrics.items() if k != \'confusion_matrix\'}}, {\'split\': \'test\', **{k: v for k, v in test_metrics.items() if k != \'confusion_matrix\'}}])\ndisplay(metrics_df.set_index(\'split\'))\npd.dataframe({\'label\': val_labels, \'score\': val_scores}).to_csv(results_dir / \'val_scores.csv\', index=false)\npd.dataframe({\'label\': test_labels, \'score\': test_scores}).to_csv(results_dir / \'test_scores.csv\', index=false)\nsummary = {\'threshold\': threshold, \'threshold_quantile\': threshold_quantile, \'val\': val_metrics, \'test\': test_metrics}\n(results_dir / \'summary.json\').write_text(json.dumps(summary, indent=2), encoding=\'utf-8\')\nprint(f"\\ntest f1={test_metrics[\'f1\']:.4f}  auroc={test_metrics[\'auroc\']:.4f}  auprc={test_metrics[\'auprc\']:.4f}")'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Score Distribution Plot

In [ ]:
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, labels, scores, split in [(axes[0], val_labels, val_scores, 'Val'), (axes[1], test_labels, test_scores, 'Test')]:
        ax.hist(scores[labels == 0], bins=80, alpha=0.6, label='Normal', density=True, color='steelblue')
        ax.hist(scores[labels == 1], bins=80, alpha=0.6, label='Defect', density=True, color='tomato')
        ax.axvline(threshold, color='black', linestyle='--', linewidth=1.2, label=f'Threshold ({threshold:.4f})')
        ax.set_title(f'{split} score distribution')
        ax.set_xlabel('Anomaly score')
        ax.set_ylabel('Density')
        ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'score_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "fig, axes = plt.subplots(1, 2, figsize=(14, 4))\nfor ax, labels, scores, split in [(axes[0], val_labels, val_scores, 'val'), (axes[1], test_labels, test_scores, 'test')]:\n    ax.hist(scores[labels == 0], bins=80, alpha=0.6, label='normal', density=true, color='steelblue')\n    ax.hist(scores[labels == 1], bins=80, alpha=0.6, label='defect', density=true, color='tomato')\n    ax.axvline(threshold, color='black', linestyle='--', linewidth=1.2, label=f'threshold ({threshold:.4f})')\n    ax.set_title(f'{split} score distribution')\n    ax.set_xlabel('anomaly score')\n    ax.set_ylabel('density')\n    ax.legend()\nplt.tight_layout()\nfig.savefig(plots_dir / 'score_distribution.png', dpi=150, bbox_inches='tight')\nplt.show()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Threshold Sweep

In [ ]:
try:
    sweep_df, best_sweep = sweep_threshold_metrics(test_labels, test_scores)
    sweep_df.to_csv(RESULTS_DIR / 'threshold_sweep.csv', index=False)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sweep_df['threshold'], sweep_df['f1'], label='F1')
    ax.plot(sweep_df['threshold'], sweep_df['precision'], label='Precision', linestyle='--')
    ax.plot(sweep_df['threshold'], sweep_df['recall'], label='Recall', linestyle=':')
    ax.axvline(threshold, color='black', linestyle='--', linewidth=1.2, label=f'Val threshold ({threshold:.4f})')
    ax.axvline(best_sweep['threshold'], color='green', linestyle=':', linewidth=1.2, label=f"Oracle best F1 ({best_sweep['f1']:.4f})")
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Test threshold sweep')
    ax.legend()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'threshold_sweep.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Val-normal threshold F1 : {test_metrics['f1']:.4f}")
    print(f"Oracle best-threshold F1: {best_sweep['f1']:.4f}")
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'sweep_df, best_sweep = sweep_threshold_metrics(test_labels, test_scores)\nsweep_df.to_csv(results_dir / \'threshold_sweep.csv\', index=false)\nfig, ax = plt.subplots(figsize=(9, 4))\nax.plot(sweep_df[\'threshold\'], sweep_df[\'f1\'], label=\'f1\')\nax.plot(sweep_df[\'threshold\'], sweep_df[\'precision\'], label=\'precision\', linestyle=\'--\')\nax.plot(sweep_df[\'threshold\'], sweep_df[\'recall\'], label=\'recall\', linestyle=\':\')\nax.axvline(threshold, color=\'black\', linestyle=\'--\', linewidth=1.2, label=f\'val threshold ({threshold:.4f})\')\nax.axvline(best_sweep[\'threshold\'], color=\'green\', linestyle=\':\', linewidth=1.2, label=f"oracle best f1 ({best_sweep[\'f1\']:.4f})")\nax.set_xlabel(\'threshold\')\nax.set_ylabel(\'score\')\nax.set_title(\'test threshold sweep\')\nax.legend()\nplt.tight_layout()\nfig.savefig(plots_dir / \'threshold_sweep.png\', dpi=150, bbox_inches=\'tight\')\nplt.show()\nprint(f"val-normal threshold f1 : {test_metrics[\'f1\']:.4f}")\nprint(f"oracle best-threshold f1: {best_sweep[\'f1\']:.4f}")'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Confusion Matrix

In [ ]:
try:
    cm = np.array(test_metrics['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    plt.colorbar(im, ax=ax)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred Normal', 'Pred Defect'])
    ax.set_yticklabels(['True Normal', 'True Defect'])
    ax.set_title('Test confusion matrix')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "cm = np.array(test_metrics['confusion_matrix'])\nfig, ax = plt.subplots(figsize=(5, 4))\nim = ax.imshow(cm, cmap='blues')\nplt.colorbar(im, ax=ax)\nfor i in range(2):\n    for j in range(2):\n        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='white' if cm[i, j] > cm.max() / 2 else 'black')\nax.set_xticks([0, 1])\nax.set_yticks([0, 1])\nax.set_xticklabels(['pred normal', 'pred defect'])\nax.set_yticklabels(['true normal', 'true defect'])\nax.set_title('test confusion matrix')\nplt.tight_layout()\nfig.savefig(plots_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')\nplt.show()"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Per-Defect Recall

In [ ]:
try:
    test_meta = metadata[metadata['split'] == 'test'].reset_index(drop=True)
    test_meta = test_meta.copy()
    test_meta['score'] = test_scores
    test_meta['predicted'] = (test_scores > threshold).astype(int)
    defect_label_col = 'failureType' if 'failureType' in test_meta.columns else None
    if defect_label_col:
        defect_rows = test_meta[test_meta['label'] == 1] if 'label' in test_meta.columns else test_meta[test_meta['split'] == 'test']
        true_defects = test_meta[test_labels == 1].copy()
        recall_by_defect = true_defects.groupby(defect_label_col).apply(lambda g: g['predicted'].mean()).rename('recall').reset_index().sort_values('recall', ascending=False)
        display(recall_by_defect)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(recall_by_defect[defect_label_col].astype(str), recall_by_defect['recall'], color='steelblue')
        ax.axhline(test_metrics['recall'], color='red', linestyle='--', label=f"Overall recall ({test_metrics['recall']:.3f})")
        ax.set_xlabel('Defect type')
        ax.set_ylabel('Recall')
        ax.set_title('Per-defect recall (test set)')
        ax.legend()
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        fig.savefig(PLOTS_DIR / 'per_defect_recall.png', dpi=150, bbox_inches='tight')
        plt.show()
        recall_by_defect.to_csv(RESULTS_DIR / 'per_defect_recall.csv', index=False)
    else:
        print('No failureType column found in metadata - skipping per-defect breakdown.')
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = 'test_meta = metadata[metadata[\'split\'] == \'test\'].reset_index(drop=true)\ntest_meta = test_meta.copy()\ntest_meta[\'score\'] = test_scores\ntest_meta[\'predicted\'] = (test_scores > threshold).astype(int)\ndefect_label_col = \'failuretype\' if \'failuretype\' in test_meta.columns else none\nif defect_label_col:\n    defect_rows = test_meta[test_meta[\'label\'] == 1] if \'label\' in test_meta.columns else test_meta[test_meta[\'split\'] == \'test\']\n    true_defects = test_meta[test_labels == 1].copy()\n    recall_by_defect = true_defects.groupby(defect_label_col).apply(lambda g: g[\'predicted\'].mean()).rename(\'recall\').reset_index().sort_values(\'recall\', ascending=false)\n    display(recall_by_defect)\n    fig, ax = plt.subplots(figsize=(10, 4))\n    ax.bar(recall_by_defect[defect_label_col].astype(str), recall_by_defect[\'recall\'], color=\'steelblue\')\n    ax.axhline(test_metrics[\'recall\'], color=\'red\', linestyle=\'--\', label=f"overall recall ({test_metrics[\'recall\']:.3f})")\n    ax.set_xlabel(\'defect type\')\n    ax.set_ylabel(\'recall\')\n    ax.set_title(\'per-defect recall (test set)\')\n    ax.legend()\n    plt.xticks(rotation=45, ha=\'right\')\n    plt.tight_layout()\n    fig.savefig(plots_dir / \'per_defect_recall.png\', dpi=150, bbox_inches=\'tight\')\n    plt.show()\n    recall_by_defect.to_csv(results_dir / \'per_defect_recall.csv\', index=false)\nelse:\n    print(\'no failuretype column found in metadata - skipping per-defect breakdown.\')'
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## UMAP Review

Loads saved embedding / UMAP artifacts by default.
to recompute them from the current checkpoint.

In [ ]:
try:
    from wafer_defect.evaluation.umap_reference import export_reference_umap_bundle
    UMAP_POINTS_PATH = UMAP_DIR / 'umap_points.csv'
    UMAP_SUMMARY_PATH = UMAP_DIR / 'umap_summary.json'
    UMAP_SPLIT_PNG = UMAP_DIR / 'umap_by_split.png'
    UMAP_SCORE_PNG = UMAP_DIR / 'umap_by_score.png'

    @torch.no_grad()
    def infer_embeddings(split: str) -> tuple[np.ndarray, np.ndarray]:
        dataset = WaferMapDataset(str(METADATA_PATH), split=split, image_size=image_size)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
        all_embeddings, all_labels = ([], [])
        for images, labels in loader:
            images = images.to(device)
            _, _, f3 = model.encode(images)
            pooled = f3.mean(dim=(2, 3)).cpu().numpy().astype(np.float32)
            all_embeddings.append(pooled)
            all_labels.append(labels.numpy())
        return (np.concatenate(all_embeddings), np.concatenate(all_labels))
    umap_ready = UMAP_POINTS_PATH.exists() and UMAP_SUMMARY_PATH.exists() and UMAP_SPLIT_PNG.exists() and UMAP_SCORE_PNG.exists()
    if not REGENERATE_UMAP and umap_ready:
        print('Loading saved UMAP artifacts ...')
    elif REGENERATE_UMAP:
        if REGENERATE_UMAP:
            if REGENERATE_UMAP:
                if REGENERATE_UMAP:
                    print('Generating UMAP artifacts from checkpoint embeddings ...')
                    import umap as umap_module
                    train_embeddings, train_embedding_labels = infer_embeddings('train')
                    val_embeddings, val_embedding_labels = infer_embeddings('val')
                    test_embeddings, test_embedding_labels = infer_embeddings('test')
                    np.save(UMAP_DIR / 'train_normal_embeddings.npy', train_embeddings)
                    np.save(UMAP_DIR / 'train_normal_labels.npy', train_embedding_labels)
                    np.save(UMAP_DIR / 'val_embeddings.npy', val_embeddings)
                    np.save(UMAP_DIR / 'val_labels.npy', val_embedding_labels)
                    np.save(UMAP_DIR / 'test_embeddings.npy', test_embeddings)
                    np.save(UMAP_DIR / 'test_labels.npy', test_embedding_labels)
                    export_reference_umap_bundle(output_dir=UMAP_DIR, umap_module=umap_module, train_normal_embeddings=train_embeddings, val_embeddings=val_embeddings, val_labels=val_embedding_labels, test_embeddings=test_embeddings, test_labels=test_embedding_labels, val_model_scores=val_scores.astype(np.float32), test_model_scores=test_scores.astype(np.float32), threshold_quantile=threshold_quantile, random_state=int(config['run'].get('seed', 42)), pca_components=50, n_neighbors=15, min_dist=0.1, knn_k=15, metric='euclidean', title_prefix='RD4AD WRN50 x224', points_filename='umap_points.csv', split_plot_filename='umap_by_split.png', score_plot_filename='umap_by_score.png', summary_filename='umap_summary.json', sweep_filename='umap_knn_threshold_sweep.csv')
                else:
                    print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
            else:
                print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
        else:
            print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    else:
        print('[WARNING] The rerun/training flags are False and the saved artifacts for this section are missing. Skipping this section.')
    umap_points_df = pd.read_csv(UMAP_POINTS_PATH)
    umap_summary = json.loads(UMAP_SUMMARY_PATH.read_text(encoding='utf-8'))
    display(pd.DataFrame([{'umap_knn_threshold': umap_summary['umap_knn_threshold'], 'umap_knn_f1': umap_summary['umap_knn_metrics']['f1'], 'umap_knn_auroc': umap_summary['umap_knn_metrics']['auroc'], 'umap_knn_auprc': umap_summary['umap_knn_metrics']['auprc']}]))
    display(umap_points_df.head())
    if UMAP_SPLIT_PNG.exists():
        display(IPyImage(filename=str(UMAP_SPLIT_PNG)))
    if UMAP_SCORE_PNG.exists():
        display(IPyImage(filename=str(UMAP_SCORE_PNG)))
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from wafer_defect.evaluation.umap_reference import export_reference_umap_bundle\numap_points_path = umap_dir / 'umap_points.csv'\numap_summary_path = umap_dir / 'umap_summary.json'\numap_split_png = umap_dir / 'umap_by_split.png'\numap_score_png = umap_dir / 'umap_by_score.png'\n\n@torch.no_grad()\ndef infer_embeddings(split: str) -> tuple[np.ndarray, np.ndarray]:\n    dataset = wafermapdataset(str(metadata_path), split=split, image_size=image_size)\n    loader = dataloader(dataset, batch_size=batch_size, shuffle=false, num_workers=num_workers)\n    all_embeddings, all_labels = ([], [])\n    for images, labels in loader:\n        images = images.to(device)\n        _, _, f3 = model.encode(images)\n        pooled = f3.mean(dim=(2, 3)).cpu().numpy().astype(np.float32)\n        all_embeddings.append(pooled)\n        all_labels.append(labels.numpy())\n    return (np.concatenate(all_embeddings), np.concatenate(all_labels))\numap_ready = umap_points_path.exists() and umap_summary_path.exists() and umap_split_png.exists() and umap_score_png.exists()\nif not regenerate_umap and umap_ready:\n    print('loading saved umap artifacts ...')\nelif regenerate_umap:\n    print('generating umap artifacts from checkpoint embeddings ...')\n    import umap as umap_module\n    train_embeddings, train_embedding_labels = infer_embeddings('train')\n    val_embeddings, val_embedding_labels = infer_embeddings('val')\n    test_embeddings, test_embedding_labels = infer_embeddings('test')\n    np.save(umap_dir / 'train_normal_embeddings.npy', train_embeddings)\n    np.save(umap_dir / 'train_normal_labels.npy', train_embedding_labels)\n    np.save(umap_dir / 'val_embeddings.npy', val_embeddings)\n    np.save(umap_dir / 'val_labels.npy', val_embedding_labels)\n    np.save(umap_dir / 'test_embeddings.npy', test_embeddings)\n    np.save(umap_dir / 'test_labels.npy', test_embedding_labels)\n    export_reference_umap_bundle(output_dir=umap_dir, umap_module=umap_module, train_normal_embeddings=train_embeddings, val_embeddings=val_embeddings, val_labels=val_embedding_labels, test_embeddings=test_embeddings, test_labels=test_embedding_labels, val_model_scores=val_scores.astype(np.float32), test_model_scores=test_scores.astype(np.float32), threshold_quantile=threshold_quantile, random_state=int(config['run'].get('seed', 42)), pca_components=50, n_neighbors=15, min_dist=0.1, knn_k=15, metric='euclidean', title_prefix='rd4ad wrn50 x224', points_filename='umap_points.csv', split_plot_filename='umap_by_split.png', score_plot_filename='umap_by_score.png', summary_filename='umap_summary.json', sweep_filename='umap_knn_threshold_sweep.csv')\nelse:\n    print('[warning] the rerun/training flags are false and the saved artifacts for this section are missing. skipping this section.')\numap_points_df = pd.read_csv(umap_points_path)\numap_summary = json.loads(umap_summary_path.read_text(encoding='utf-8'))\ndisplay(pd.dataframe([{'umap_knn_threshold': umap_summary['umap_knn_threshold'], 'umap_knn_f1': umap_summary['umap_knn_metrics']['f1'], 'umap_knn_auroc': umap_summary['umap_knn_metrics']['auroc'], 'umap_knn_auprc': umap_summary['umap_knn_metrics']['auprc']}]))\ndisplay(umap_points_df.head())\nif umap_split_png.exists():\n    display(ipyimage(filename=str(umap_split_png)))\nif umap_score_png.exists():\n    display(ipyimage(filename=str(umap_score_png)))"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Saved Outputs

In [ ]:
try:
    saved = {'artifact_dir': str(ARTIFACT_DIR), 'results_dir': str(RESULTS_DIR), 'plots_dir': str(PLOTS_DIR), 'umap_dir': str(UMAP_DIR), 'checkpoint': str(CHECKPOINT_PATH), 'threshold': threshold, 'test_f1': test_metrics['f1'], 'test_auroc': test_metrics['auroc'], 'test_auprc': test_metrics['auprc']}
    saved
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "saved = {'artifact_dir': str(artifact_dir), 'results_dir': str(results_dir), 'plots_dir': str(plots_dir), 'umap_dir': str(umap_dir), 'checkpoint': str(checkpoint_path), 'threshold': threshold, 'test_f1': test_metrics['f1'], 'test_auroc': test_metrics['auroc'], 'test_auprc': test_metrics['auprc']}\nsaved"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
